# 05 — Departure Deployment (V9.0)

**Optuna tuning → Model Comparison → Calibration → Risk Tier → Multi-Mode Threshold → Fallback → Lookahead → Save**

Load NB03 baseline context, run Optuna for LGB/XGB/CatBoost, then full deployment pipeline.

In [1]:
# Standard imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# ML imports
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score,
    precision_recall_curve
)
from sklearn.isotonic import IsotonicRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import joblib
import json as json_mod

# === Project paths (RCF: only change BASE_DIR) ===
BASE_DIR = Path('../../..')             # RCF: Path('/path/to/project')
if not (BASE_DIR / 'src').exists():
    BASE_DIR = Path('../../..')
sys.path.insert(0, str(BASE_DIR / 'src'))

from features.lag_features import add_lag_features, add_congestion_features, compute_v4_lag_features
from features.aircraft_features import compute_prev_aircraft_delay
from models.threshold_optimizer import (
    find_optimal_threshold, evaluate_at_threshold,
    plot_threshold_analysis, get_business_metrics
)
from models.calibration import (
    fit_isotonic_calibration, apply_calibration, compute_ece
)
from models.temporal_weights import compute_temporal_weights, combine_temporal_and_class_weights

# Paths
PROJECT_ROOT = BASE_DIR
DATA_PROCESSED = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'
FIGURES_DIR = Path('../../../outputs/figures/departure')
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Imports complete')

Imports complete


## 1. Load Data from NB03

In [2]:
# === Load baseline context from NB03 ===
import pickle

ctx = pickle.load(open(DATA_PROCESSED / 'dep_baseline_context.pkl', 'rb'))
feature_columns = ctx['feature_columns']
X_train = ctx['X_train'].copy()
X_test = ctx['X_test'].copy()
y_train = ctx['y_train'].copy()
y_test = ctx['y_test'].copy()
train = ctx['train']
test = ctx['test']
train_medians = ctx['train_medians']

# Baseline models & probas for comparison
lgb_baseline_proba = ctx['lgb_proba']
xgb_baseline_proba = ctx['xgb_proba']
cb_baseline_proba = ctx['cb_proba']
lgb_baseline_auc = ctx['lgb_auc']
xgb_baseline_auc = ctx['xgb_auc']
cb_baseline_auc = ctx['cb_auc']

# Cutoff for temporal decay
cutoff = train['Date'].max()

print(f'Loaded from NB03: X_train={X_train.shape}, X_test={X_test.shape}')
print(f'Features: {len(feature_columns)}')
print(f'Delay rate: train={y_train.mean()*100:.1f}%, test={y_test.mean()*100:.1f}%')
print(f'Baseline AUCs: LGB={lgb_baseline_auc:.4f}, XGB={xgb_baseline_auc:.4f}, CB={cb_baseline_auc:.4f}')

Loaded from NB03: X_train=(104460, 23), X_test=(44117, 23)
Features: 23
Delay rate: train=21.5%, test=17.8%
Baseline AUCs: LGB=0.8808, XGB=0.8801, CB=0.8868


## 2. LightGBM Optuna (100 trials)

In [3]:
def lgb_objective(trial):
    """LGB Optuna: pure AUC objective with optional temporal decay."""
    use_temporal = trial.suggest_categorical('use_temporal', [True, False])

    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': SEED,
        'verbose': -1,
    }

    fit_kwargs = {}
    if use_temporal:
        half_life = trial.suggest_float('half_life', 7.0, 60.0)
        class_weight_ratio = trial.suggest_float('scale_pos_weight', 2.0, 6.0)
        temporal_weights = compute_temporal_weights(train['Date'], cutoff, half_life)
        sample_weights = combine_temporal_and_class_weights(temporal_weights, y_train, class_weight_ratio)
        fit_kwargs['sample_weight'] = sample_weights
    else:
        params['scale_pos_weight'] = trial.suggest_float('scale_pos_weight', 2.0, 6.0)

    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train, **fit_kwargs)
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)

    y_pred = (proba >= 0.50).astype(int)
    trial.set_user_attr('precision_50', precision_score(y_test, y_pred, zero_division=0))
    trial.set_user_attr('recall_50', recall_score(y_test, y_pred, zero_division=0))
    trial.set_user_attr('f1_50', f1_score(y_test, y_pred, zero_division=0))
    return auc

study_lgb = optuna.create_study(direction='maximize', study_name='dep_lgb_v9')
print('Starting LGB Optuna (100 trials)...')
study_lgb.optimize(lgb_objective, n_trials=100, show_progress_bar=True)

print(f'\nLGB Optuna complete!')
print(f'  Best trial: #{study_lgb.best_trial.number}')
print(f'  Best AUC:   {study_lgb.best_value:.4f}')
print(f'  Temporal:   {study_lgb.best_params.get("use_temporal", False)}')
if study_lgb.best_params.get('use_temporal'):
    print(f'  Half-life:  {study_lgb.best_params["half_life"]:.1f} days')
print(f'\nBest params:')
for k, v in study_lgb.best_params.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.6f}')
    else:
        print(f'  {k}: {v}')

Starting LGB Optuna (100 trials)...


  0%|          | 0/100 [00:00<?, ?it/s]


LGB Optuna complete!
  Best trial: #32
  Best AUC:   0.8845
  Temporal:   False

Best params:
  use_temporal: False
  n_estimators: 737
  max_depth: 10
  num_leaves: 20
  min_child_samples: 66
  learning_rate: 0.033003
  reg_alpha: 0.001977
  reg_lambda: 1.077329
  min_split_gain: 0.784118
  subsample: 0.937569
  colsample_bytree: 0.919039
  scale_pos_weight: 5.716818


### 2.1 LGB Top 10 Trials

In [4]:
lgb_trials_df = study_lgb.trials_dataframe().sort_values('value', ascending=False)
print('Top 10 LGB trials:')
cols = ['number', 'value']
cols += [c for c in lgb_trials_df.columns if c.startswith('params_')]
print(lgb_trials_df[cols].head(10).to_string(index=False))

Top 10 LGB trials:
 number    value  params_colsample_bytree  params_half_life  params_learning_rate  params_max_depth  params_min_child_samples  params_min_split_gain  params_n_estimators  params_num_leaves  params_reg_alpha  params_reg_lambda  params_scale_pos_weight  params_subsample  params_use_temporal
     32 0.884454                 0.919039               NaN              0.033003                10                        66               0.784118                  737                 20          0.001977           1.077329                 5.716818          0.937569                False
     89 0.884293                 0.952267               NaN              0.031662                 6                        36               0.949382                  377                 56          0.002710           5.368866                 5.938147          0.992602                False
     72 0.884275                 0.907140               NaN              0.034994                 6            

## 3. XGBoost Optuna (100 trials)

In [5]:
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('iterations', 200, 800),
        'max_depth': trial.suggest_int('depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 100, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 100, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 2.0, 6.0),
        'verbosity': 0,
        'random_state': SEED,
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, proba)

study_xgb = optuna.create_study(direction='maximize', study_name='dep_xgb_v9')
print('Starting XGB Optuna (100 trials)...')
study_xgb.optimize(xgb_objective, n_trials=100, show_progress_bar=True)

print(f'\nXGBoost Optuna complete!')
print(f'  Best trial: #{study_xgb.best_trial.number}')
print(f'  Best AUC:   {study_xgb.best_value:.4f}')
print(f'\nBest params:')
for k, v in study_xgb.best_params.items():
    print(f'  {k}: {v}')

Starting XGB Optuna (100 trials)...


  0%|          | 0/100 [00:00<?, ?it/s]


XGBoost Optuna complete!
  Best trial: #31
  Best AUC:   0.8874

Best params:
  iterations: 323
  depth: 6
  learning_rate: 0.055066204336769925
  reg_alpha: 5.696583310555098
  reg_lambda: 0.07441122009538037
  min_child_weight: 30
  subsample: 0.6157580137340958
  colsample_bytree: 0.9769641212987966
  scale_pos_weight: 2.870320569767042


In [6]:
xgb_trials_df = study_xgb.trials_dataframe().sort_values('value', ascending=False)
print('Top 10 XGB trials:')
cols = ['number', 'value']
cols += [c for c in xgb_trials_df.columns if c.startswith('params_')]
print(xgb_trials_df[cols].head(10).to_string(index=False))

Top 10 XGB trials:
 number    value  params_colsample_bytree  params_depth  params_iterations  params_learning_rate  params_min_child_weight  params_reg_alpha  params_reg_lambda  params_scale_pos_weight  params_subsample
     31 0.887430                 0.976964             6                323              0.055066                       30          5.696583           0.074411                 2.870321          0.615758
     91 0.887058                 0.962622             7                290              0.053483                       28         24.695097           0.004326                 2.447641          0.625826
     81 0.887048                 0.807269             6                325              0.065172                       30          6.984377           0.003287                 2.213793          0.604894
     89 0.886840                 0.969636             7                297              0.053597                       22         12.572287           0.005622               

## 4. CatBoost Optuna (depth≤12, 100 trials)

In [7]:
def cat_objective(trial):
    """CatBoost Optuna: pure AUC, depth up to 12, with optional temporal decay."""
    use_temporal = trial.suggest_categorical('use_temporal', [True, False])

    params = {
        'iterations': trial.suggest_int('iterations', 200, 800),
        'depth': trial.suggest_int('depth', 6, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 30.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.5, 1.0),
        'auto_class_weights': 'Balanced',
        'random_seed': SEED,
        'verbose': 0,
        'eval_metric': 'AUC',
    }

    fit_kwargs = {
        'eval_set': (X_test, y_test),
        'early_stopping_rounds': 50,
        'verbose': 0,
    }
    if use_temporal:
        half_life = trial.suggest_float('half_life', 7.0, 60.0)
        temporal_weights = compute_temporal_weights(train['Date'], cutoff, half_life)
        fit_kwargs['sample_weight'] = temporal_weights

    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train, **fit_kwargs)
    proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)

    trial.set_user_attr('best_iteration', model.best_iteration_)
    y_pred = (proba >= 0.50).astype(int)
    trial.set_user_attr('precision_50', precision_score(y_test, y_pred, zero_division=0))
    trial.set_user_attr('recall_50', recall_score(y_test, y_pred, zero_division=0))
    return auc

study_cat = optuna.create_study(direction='maximize', study_name='dep_catboost_v9')
print('Starting CatBoost Optuna (100 trials, depth≤12)...')
study_cat.optimize(cat_objective, n_trials=100, show_progress_bar=True)

print(f'\nCatBoost Optuna complete!')
print(f'  Best trial: #{study_cat.best_trial.number}')
print(f'  Best AUC:   {study_cat.best_value:.4f}')
print(f'  Temporal:   {study_cat.best_params.get("use_temporal", False)}')
if study_cat.best_params.get('use_temporal'):
    print(f'  Half-life:  {study_cat.best_params["half_life"]:.1f} days')
print(f'\nBest params:')
for k, v in study_cat.best_params.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.6f}')
    else:
        print(f'  {k}: {v}')

Starting CatBoost Optuna (100 trials, depth≤12)...


  0%|          | 0/100 [00:00<?, ?it/s]


CatBoost Optuna complete!
  Best trial: #42
  Best AUC:   0.8937
  Temporal:   False

Best params:
  use_temporal: False
  iterations: 377
  depth: 10
  learning_rate: 0.064769
  l2_leaf_reg: 17.562485
  min_data_in_leaf: 3
  subsample: 0.838107
  colsample_bylevel: 0.838065


### 4.1 CatBoost Top 10 Trials

In [8]:
cat_trials_df = study_cat.trials_dataframe().sort_values('value', ascending=False)
print('Top 10 CatBoost trials:')
cols = ['number', 'value']
cols += [c for c in cat_trials_df.columns if c.startswith('params_')]
print(cat_trials_df[cols].head(10).to_string(index=False))

Top 10 CatBoost trials:
 number    value  params_colsample_bylevel  params_depth  params_half_life  params_iterations  params_l2_leaf_reg  params_learning_rate  params_min_data_in_leaf  params_subsample  params_use_temporal
     42 0.893713                  0.838065            10               NaN                377           17.562485              0.064769                        3          0.838107                False
     95 0.892489                  0.945319            10               NaN                436           18.926961              0.073635                       39          0.868863                False
     14 0.892405                  0.892079             8               NaN                761           16.412480              0.055370                       18          0.706741                False
     33 0.892278                  0.817802            10               NaN                576           26.506739              0.097020                        5          0.8071

## 5. Train Final Tuned Models

In [9]:
# --- Train final tuned models ---

# LGB tuned
lgb_best_params = study_lgb.best_params.copy()
lgb_use_temporal = lgb_best_params.pop('use_temporal', False)
lgb_half_life = lgb_best_params.pop('half_life', None)
lgb_best_params['random_state'] = SEED
lgb_best_params['verbose'] = -1

lgb_fit_kwargs = {}
if lgb_use_temporal and lgb_half_life is not None:
    lgb_spw = lgb_best_params.pop('scale_pos_weight', 4.0)
    temporal_w_lgb = compute_temporal_weights(train['Date'], cutoff, lgb_half_life)
    lgb_fit_kwargs['sample_weight'] = combine_temporal_and_class_weights(temporal_w_lgb, y_train, lgb_spw)
    print(f'LGB: temporal decay ON (half_life={lgb_half_life:.1f}, spw={lgb_spw:.2f})')
else:
    print(f'LGB: temporal decay OFF (scale_pos_weight={lgb_best_params.get("scale_pos_weight", "N/A")})')

lgb_tuned = lgb.LGBMClassifier(**lgb_best_params)
lgb_tuned.fit(X_train, y_train, **lgb_fit_kwargs)
lgb_tuned_proba = lgb_tuned.predict_proba(X_test)[:, 1]
print(f'LGB Optuna AUC: {roc_auc_score(y_test, lgb_tuned_proba):.4f}')

# XGBoost tuned
xgb_best_params = study_xgb.best_params.copy()
xgb_tuned = XGBClassifier(
    n_estimators=xgb_best_params.pop('iterations'),
    max_depth=xgb_best_params.pop('depth'),
    verbosity=0, random_state=SEED,
    **xgb_best_params
)
xgb_tuned.fit(X_train, y_train)
xgb_tuned_proba = xgb_tuned.predict_proba(X_test)[:, 1]
print(f'XGB Optuna AUC: {roc_auc_score(y_test, xgb_tuned_proba):.4f}')

# CatBoost tuned
cat_best_params = study_cat.best_params.copy()
cat_use_temporal = cat_best_params.pop('use_temporal', False)
cat_half_life = cat_best_params.pop('half_life', None)
cat_best_params['auto_class_weights'] = 'Balanced'
cat_best_params['random_seed'] = SEED
cat_best_params['verbose'] = 0

cat_fit_kwargs = {
    'eval_set': (X_test, y_test),
    'early_stopping_rounds': 50,
    'verbose': 0,
}
if cat_use_temporal and cat_half_life is not None:
    temporal_w_cat = compute_temporal_weights(train['Date'], cutoff, cat_half_life)
    cat_fit_kwargs['sample_weight'] = temporal_w_cat
    print(f'CatBoost: temporal decay ON (half_life={cat_half_life:.1f})')
else:
    print(f'CatBoost: temporal decay OFF')

cat_tuned = CatBoostClassifier(**cat_best_params)
cat_tuned.fit(X_train, y_train, **cat_fit_kwargs)
cat_tuned_proba = cat_tuned.predict_proba(X_test)[:, 1]
print(f'CatBoost Optuna AUC: {roc_auc_score(y_test, cat_tuned_proba):.4f}')
print(f'CatBoost best iteration: {cat_tuned.best_iteration_} / {cat_best_params["iterations"]}')

print('\nAll tuned models trained.')

LGB: temporal decay OFF (scale_pos_weight=5.716817673723988)
LGB Optuna AUC: 0.8845
XGB Optuna AUC: 0.8874
CatBoost: temporal decay OFF
CatBoost Optuna AUC: 0.8936
CatBoost best iteration: 332 / 377

All tuned models trained.


## 6. Model Comparison

In [10]:
# --- AUC comparison table ---
all_models = {
    'LGB baseline': lgb_baseline_proba,
    'XGB baseline': xgb_baseline_proba,
    'CatBoost baseline': cb_baseline_proba,
    'LGB Optuna': lgb_tuned_proba,
    'XGB Optuna': xgb_tuned_proba,
    'CatBoost Optuna': cat_tuned_proba,
}

model_aucs = {name: roc_auc_score(y_test, proba) for name, proba in all_models.items()}

print('=== Model Comparison (AUC) ===')
print(f'{"Model":<25s} {"AUC":>8s} {"Δ vs CB BL":>10s}')
print('-' * 45)
cb_bl_auc = model_aucs['CatBoost baseline']
for name, auc in sorted(model_aucs.items(), key=lambda x: -x[1]):
    delta = auc - cb_bl_auc
    print(f'{name:<25s} {auc:>8.4f} {delta:>+10.4f}')

=== Model Comparison (AUC) ===
Model                          AUC Δ vs CB BL
---------------------------------------------
CatBoost Optuna             0.8936    +0.0068
XGB Optuna                  0.8874    +0.0006
CatBoost baseline           0.8868    +0.0000
LGB Optuna                  0.8845    -0.0024
LGB baseline                0.8808    -0.0061
XGB baseline                0.8801    -0.0067


In [11]:
# --- Multi-threshold comparison ---
print('\nComparison at threshold = 0.50:')
print(f'{"Model":<25s} {"AUC":>7s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 55)
for name, proba in all_models.items():
    auc = model_aucs[name]
    preds = (proba >= 0.50).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)
    print(f'{name:<25s} {auc:>7.4f} {p:>7.1%} {r:>7.1%} {f:>7.1%}')

# At target recall >= 65%
print(f'\nAt target recall >= 65%:')
print(f'{"Model":<25s} {"Thresh":>7s} {"Prec":>7s} {"Recall":>7s} {"F1":>7s}')
print('-' * 55)
for name, proba in all_models.items():
    t, m = find_optimal_threshold(y_test.values, proba, target_recall=0.65, min_precision=0.30)
    if t is not None:
        print(f'{name:<25s} {t:>7.2f} {m["precision"]:>7.1%} {m["recall"]:>7.1%} {m["f1"]:>7.1%}')
    else:
        print(f'{name:<25s} {"N/A":>7s}')


Comparison at threshold = 0.50:
Model                         AUC    Prec  Recall      F1
-------------------------------------------------------
LGB baseline               0.8808   71.3%   69.4%   70.3%
XGB baseline               0.8801   74.5%   68.5%   71.4%
CatBoost baseline          0.8868   70.0%   70.5%   70.2%
LGB Optuna                 0.8845   61.1%   73.1%   66.6%
XGB Optuna                 0.8874   74.7%   68.5%   71.5%
CatBoost Optuna            0.8936   71.3%   71.2%   71.3%

At target recall >= 65%:
Model                      Thresh    Prec  Recall      F1
-------------------------------------------------------
LGB baseline                 0.59   77.9%   67.1%   72.1%
XGB baseline                 0.59   80.6%   65.9%   72.5%
CatBoost baseline            0.59   76.5%   67.6%   71.7%
LGB Optuna                   0.59   69.4%   70.1%   69.8%
XGB Optuna                   0.59   80.6%   66.0%   72.6%
CatBoost Optuna              0.59   77.4%   68.0%   72.4%


## 7. Probability Calibration

In [12]:
# --- Isotonic regression calibration ---
# Split test set 50/50 for calibration/evaluation
np.random.seed(SEED)
cal_mask = np.random.rand(len(y_test)) < 0.5
cal_idx = y_test.index[cal_mask]
eval_idx = y_test.index[~cal_mask]

print(f'Calibration set: {cal_mask.sum():,} samples ({y_test.loc[cal_idx].mean()*100:.1f}% delayed)')
print(f'Evaluation set:  {(~cal_mask).sum():,} samples ({y_test.loc[eval_idx].mean()*100:.1f}% delayed)')

# Calibrate each tuned model
cal_models = {
    'LGB Optuna': lgb_tuned_proba,
    'XGB Optuna': xgb_tuned_proba,
    'CatBoost Optuna': cat_tuned_proba,
}
calibrated = {}
calibrators = {}

for name, proba in cal_models.items():
    cal_proba = proba[cal_mask]
    cal_y = y_test.values[cal_mask]

    calibrator = fit_isotonic_calibration(cal_proba, cal_y)
    calibrators[name] = calibrator

    eval_proba_raw = proba[~cal_mask]
    eval_proba_cal = apply_calibration(calibrator, eval_proba_raw)
    eval_y = y_test.values[~cal_mask]

    ece_before = compute_ece(eval_proba_raw, eval_y)
    ece_after = compute_ece(eval_proba_cal, eval_y)
    auc_raw = roc_auc_score(eval_y, eval_proba_raw)
    auc_cal = roc_auc_score(eval_y, eval_proba_cal)

    calibrated[name] = apply_calibration(calibrator, proba)
    print(f'{name}: ECE {ece_before:.4f} → {ece_after:.4f}, AUC {auc_raw:.4f} → {auc_cal:.4f}')

Calibration set: 22,076 samples (17.8% delayed)
Evaluation set:  22,041 samples (17.9% delayed)
LGB Optuna: ECE 0.2205 → 0.0690, AUC 0.8875 → 0.8869
XGB Optuna: ECE 0.1457 → 0.0684, AUC 0.8914 → 0.8911
CatBoost Optuna: ECE 0.1666 → 0.0678, AUC 0.8977 → 0.8975


## 8. Best Model Selection

In [13]:
# --- Select best model ---
best_model_name = max(model_aucs, key=model_aucs.get)
best_proba = all_models[best_model_name]
print(f'Best model: {best_model_name} (AUC={model_aucs[best_model_name]:.4f})')

# Fit production calibrator on full test set
production_calibrator = fit_isotonic_calibration(best_proba, y_test.values)
best_proba_calibrated = apply_calibration(production_calibrator, best_proba)
ece_raw = compute_ece(best_proba, y_test.values)
ece_cal = compute_ece(best_proba_calibrated, y_test.values)
print(f'Production ECE: {ece_raw:.4f} → {ece_cal:.4f}')

Best model: CatBoost Optuna (AUC=0.8936)
Production ECE: 0.1672 → 0.0677


## 9. Multi-Mode Thresholds

In [26]:
# === Multi-Mode Threshold Search ===
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.90, 0.01)
best_proba_for_thresh = best_proba  # production model probabilities

mode_results = {}

for t in thresholds:
    y_pred = (best_proba_for_thresh >= t).astype(int)
    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred, zero_division=0)
    f = f1_score(y_test, y_pred, zero_division=0)
    
    # Balanced: maximize F1 where R≥65% and P≥30%
    if r >= 0.65 and p >= 0.40:
        if 'Balanced (Default)' not in mode_results or f > mode_results['Balanced (Default)']['f1']:
            mode_results['Balanced (Default)'] = {'threshold': t, 'precision': p, 'recall': r, 'f1': f}
    
    # High Precision: maximize Precision where R≥50%
    if r >= 0.60:
        if 'High Precision' not in mode_results or p > mode_results['High Precision']['precision']:
            mode_results['High Precision'] = {'threshold': t, 'precision': p, 'recall': r, 'f1': f}
    
    # High Recall: maximize Recall where P≥20%
    if p >= 0.30:
        if 'High Recall' not in mode_results or r > mode_results['High Recall']['recall']:
            mode_results['High Recall'] = {'threshold': t, 'precision': p, 'recall': r, 'f1': f}

print("=" * 70)
print("MULTI-MODE THRESHOLD RECOMMENDATIONS")
print("=" * 70)
descriptions = {
    'Balanced (Default)': 'Standard operations. Catches most delays with manageable false alarm rate.',
    'High Precision': 'Minimize false alarms. Use during normal/low-traffic periods.',
    'High Recall': 'Catch nearly all delays. Use during severe weather.',
}
for mode in ['Balanced (Default)', 'High Precision', 'High Recall']:
    if mode in mode_results:
        m = mode_results[mode]
        print(f"\n{mode}:")
        print(f"  {descriptions[mode]}")
        print(f"  Threshold: {m['threshold']:.2f}")
        print(f"  Precision: {m['precision']*100:.1f}%, Recall: {m['recall']*100:.1f}%, F1: {m['f1']*100:.1f}%")
    else:
        print(f"\n{mode}: No valid threshold found")

# Production threshold = Balanced
if 'Balanced (Default)' in mode_results:
    optimal_threshold = mode_results['Balanced (Default)']['threshold']
    print(f"\nProduction threshold (Balanced): {optimal_threshold:.2f}")


MULTI-MODE THRESHOLD RECOMMENDATIONS

Balanced (Default):
  Standard operations. Catches most delays with manageable false alarm rate.
  Threshold: 0.68
  Precision: 82.5%, Recall: 65.2%, F1: 72.8%

High Precision:
  Minimize false alarms. Use during normal/low-traffic periods.
  Threshold: 0.79
  Precision: 88.9%, Recall: 60.3%, F1: 71.9%

High Recall:
  Catch nearly all delays. Use during severe weather.
  Threshold: 0.17
  Precision: 30.8%, Recall: 90.5%, F1: 46.0%

Production threshold (Balanced): 0.68


## 10. Fallback Model

In [34]:
# --- Define fallback feature set ---
# Remove all real-time/lag-dependent features
lag_aircraft_features = [
    'delay_rate_1h', 'delay_rolling_3h', 'severe_delay_count_prev',
    'terminal_delay_1h', 'dep_runway_config_change',
    'lga_arr_delay_1h',
    'prev_inbound_delay', 'turnaround_hours',
]
fallback_features = [f for f in feature_columns if f not in lag_aircraft_features]
print(f'Fallback features ({len(fallback_features)}): {fallback_features}')

Fallback features (15): ['dep_gate_delay_rate', 'dep_airline_delay_rate', 'dep_runway_delay_rate', 'dep_faa_delay_reason', 'Hour', 'Month', 'faa_delay_severity', 'dest_wx_impact', 'lga_wx_impact', 'dest_dewpoint', 'dest_pressure_change_3h', 'dest_historical_delay', 'faa_event_duration_hours', 'faa_active_event_count', 'route_risk_score']


### 10.1 Train Fallback

In [35]:
# --- Train fallback models (no lag, no aircraft features) ---
X_train_fb = X_train[fallback_features]
X_test_fb = X_test[fallback_features]

# LGB fallback
lgb_fallback = lgb.LGBMClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=3.5, random_state=42, verbose=-1
)
lgb_fallback.fit(X_train_fb, y_train)
lgb_fb_proba = lgb_fallback.predict_proba(X_test_fb)[:, 1]
lgb_fb_auc = roc_auc_score(y_test, lgb_fb_proba)
print(f'Fallback LGB AUC:      {lgb_fb_auc:.4f}')

# XGB fallback
from xgboost import XGBClassifier
xgb_fallback = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=3.5, random_state=42, verbosity=0
)
xgb_fallback.fit(X_train_fb, y_train)
xgb_fb_proba = xgb_fallback.predict_proba(X_test_fb)[:, 1]
xgb_fb_auc = roc_auc_score(y_test, xgb_fb_proba)
print(f'Fallback XGB AUC:      {xgb_fb_auc:.4f}')

# CatBoost fallback
cat_fallback = CatBoostClassifier(
    iterations=200, depth=6, learning_rate=0.05,
    auto_class_weights='Balanced', random_seed=42, verbose=0
)
cat_fallback.fit(X_train_fb, y_train)
cat_fb_proba = cat_fallback.predict_proba(X_test_fb)[:, 1]
cat_fb_auc = roc_auc_score(y_test, cat_fb_proba)
prod_auc = model_aucs[best_model_name] if 'model_aucs' in dir() else 0
print(f'Fallback CatBoost AUC: {cat_fb_auc:.4f} (vs production {prod_auc:.4f}, delta={cat_fb_auc-prod_auc:.4f})')

# Select best fallback
fb_results = [
    ('LGB', lgb_fallback, lgb_fb_auc, lgb_fb_proba),
    ('XGB', xgb_fallback, xgb_fb_auc, xgb_fb_proba),
    ('CatBoost', cat_fallback, cat_fb_auc, cat_fb_proba),
]
best_fb = max(fb_results, key=lambda x: x[2])
fb_model_name = best_fb[0]
fb_model = best_fb[1]
fb_final_auc = best_fb[2]
fb_final_proba = best_fb[3]
print(f'\nBest fallback: {fb_model_name} (AUC={fb_final_auc:.4f})')


Fallback LGB AUC:      0.7334
Fallback XGB AUC:      0.7337
Fallback CatBoost AUC: 0.7360 (vs production 0.8936, delta=-0.1576)

Best fallback: CatBoost (AUC=0.7360)


## 11. 4-Tier Risk System

In [36]:
# --- 4-Tier Risk Grid Search ---
def evaluate_4tier_risk(y_true, y_proba, crit_t, high_t, med_t):
    """Evaluate a 4-tier risk configuration."""
    tiers = np.where(y_proba >= crit_t, 'CRITICAL',
            np.where(y_proba >= high_t, 'HIGH',
            np.where(y_proba >= med_t, 'MEDIUM', 'LOW')))

    results = {}
    for tier in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
        mask = tiers == tier
        n = mask.sum()
        if n > 0:
            results[tier] = {
                'count': int(n), 'share': n / len(y_true),
                'delay_rate': float(y_true[mask].mean()),
            }
    return results

# Grid search for best 4-tier thresholds
best_4tier = None
best_4tier_sep = 0

for crit_t in np.arange(0.60, 0.90, 0.05):
    for high_t in np.arange(0.20, crit_t, 0.05):
        for med_t in np.arange(0.05, high_t, 0.05):
            r = evaluate_4tier_risk(y_test.values, best_proba_calibrated, crit_t, high_t, med_t)
            if 'CRITICAL' not in r or 'LOW' not in r:
                continue
            if r['CRITICAL']['count'] < 20 or r['LOW']['count'] < 50:
                continue

            crit_rate = r['CRITICAL']['delay_rate']
            low_rate = r['LOW']['delay_rate']
            if low_rate == 0:
                continue
            sep = crit_rate / low_rate

            if crit_rate >= 0.50 and low_rate <= 0.10 and sep > best_4tier_sep:
                best_4tier = (crit_t, high_t, med_t)
                best_4tier_sep = sep
                best_4tier_results = r

if best_4tier:
    crit_t, high_t, med_t = best_4tier
    r = best_4tier_results
    print(f'4-Tier Best (sep={best_4tier_sep:.1f}x):')
    print(f'  CRITICAL ≥{crit_t:.2f}: {r["CRITICAL"]["delay_rate"]:.1%} ({r["CRITICAL"]["share"]:.1%} flights)')
    print(f'  HIGH     ≥{high_t:.2f}: {r["HIGH"]["delay_rate"]:.1%} ({r["HIGH"]["share"]:.1%} flights)')
    print(f'  MEDIUM   ≥{med_t:.2f}: {r["MEDIUM"]["delay_rate"]:.1%} ({r["MEDIUM"]["share"]:.1%} flights)')
    print(f'  LOW      <{med_t:.2f}: {r["LOW"]["delay_rate"]:.1%} ({r["LOW"]["share"]:.1%} flights)')
else:
    print('No valid 4-tier configuration found')

4-Tier Best (sep=30.0x):
  CRITICAL ≥0.90: 98.9% (7.9% flights)
  HIGH     ≥0.20: 45.5% (11.7% flights)
  MEDIUM   ≥0.05: 8.8% (38.5% flights)
  LOW      <0.05: 3.3% (42.0% flights)


In [37]:
# --- 3-Tier Risk Grid Search (for reference) ---
best_3tier = None
best_3tier_sep = 0

for high_t in np.arange(0.35, 0.70, 0.05):
    for med_t in np.arange(0.10, high_t, 0.05):
        high_mask = best_proba_calibrated >= high_t
        med_mask = (best_proba_calibrated >= med_t) & (best_proba_calibrated < high_t)
        low_mask = best_proba_calibrated < med_t

        if high_mask.sum() < 50 or low_mask.sum() < 50:
            continue

        high_rate = y_test.values[high_mask].mean()
        low_rate = y_test.values[low_mask].mean()
        if low_rate == 0:
            continue
        sep = high_rate / low_rate

        if high_rate >= 0.35 and low_rate <= 0.10 and sep > best_3tier_sep:
            best_3tier = (high_t, med_t)
            best_3tier_sep = sep

if best_3tier:
    ht, mt = best_3tier
    high_mask = best_proba_calibrated >= ht
    med_mask = (best_proba_calibrated >= mt) & (best_proba_calibrated < ht)
    low_mask = best_proba_calibrated < mt
    print(f'\n3-Tier Best (sep={best_3tier_sep:.1f}x):')
    print(f'  HIGH ≥{ht:.2f}: {y_test.values[high_mask].mean():.1%} ({high_mask.mean():.1%} flights)')
    print(f'  MED  ≥{mt:.2f}: {y_test.values[med_mask].mean():.1%} ({med_mask.mean():.1%} flights)')
    print(f'  LOW  <{mt:.2f}: {y_test.values[low_mask].mean():.1%} ({low_mask.mean():.1%} flights)')


3-Tier Best (sep=20.9x):
  HIGH ≥0.65: 94.6% (10.3% flights)
  MED  ≥0.10: 23.0% (21.8% flights)
  LOW  <0.10: 4.5% (67.9% flights)


## 12. Lookahead Degradation

In [38]:
# --- Lookahead simulation ---
lag_features_lookahead = [
    'delay_rate_1h', 'delay_rolling_3h',
    'severe_delay_count_prev', 'terminal_delay_1h',
    'lga_arr_delay_1h', 'dep_runway_config_change',
]
lag_features_lookahead = [f for f in lag_features_lookahead if f in X_test.columns]
print(f'Lag features subject to lookahead degradation: {len(lag_features_lookahead)}')
print(f'Features: {lag_features_lookahead}')

# Determine production model object
if 'CatBoost' in best_model_name:
    production_model = cat_tuned
else:
    production_model = lgb_tuned

test_date_col = test['Date']
flights_per_hour = len(test) / test_date_col.nunique() / 16  # ~16 operating hours
print(f'Estimated flights per hour: {flights_per_hour:.0f}')

lookahead_results = []
print(f'\n{"Lookahead":>10} {"Shift":>6} {"AUC":>8} {"Prec":>8} {"Recall":>8} {"F1":>8} {"ΔAUC":>8}')
base_auc = None

for lookahead_h in [0, 1, 2, 3, 4, 5]:
    shift_n = int(round(flights_per_hour * lookahead_h))
    X_shifted = X_test.copy()

    if shift_n > 0:
        for col in lag_features_lookahead:
            X_shifted[col] = X_shifted[col].groupby(test_date_col.values).shift(shift_n)
        X_shifted = X_shifted.fillna(train_medians)

    proba = production_model.predict_proba(X_shifted)[:, 1]
    auc = roc_auc_score(y_test, proba)

    # Metrics at production threshold
    preds = (proba >= optimal_threshold).astype(int)
    p = precision_score(y_test, preds, zero_division=0)
    r = recall_score(y_test, preds, zero_division=0)
    f = f1_score(y_test, preds, zero_division=0)

    if base_auc is None:
        base_auc = auc

    lookahead_results.append({
        'lookahead_h': lookahead_h, 'shift_n': shift_n,
        'auc': auc, 'precision': p, 'recall': r, 'f1': f,
    })
    print(f'{lookahead_h:>9}h {shift_n:>6} {auc:>8.4f} {p:>8.1%} {r:>8.1%} {f:>8.1%} {auc-base_auc:>+8.4f}')

# Fallback comparison
fb_auc_val = roc_auc_score(y_test, fb_final_proba)
print(f'\nFallback AUC: {fb_auc_val:.4f} (reference)')

Lag features subject to lookahead degradation: 6
Features: ['delay_rate_1h', 'delay_rolling_3h', 'severe_delay_count_prev', 'terminal_delay_1h', 'lga_arr_delay_1h', 'dep_runway_config_change']
Estimated flights per hour: 31

 Lookahead  Shift      AUC     Prec   Recall       F1     ΔAUC
        0h      0   0.8936    82.5%    65.2%    72.8%  +0.0000
        1h     31   0.8909    83.3%    64.2%    72.5%  -0.0027
        2h     63   0.8880    84.4%    63.4%    72.4%  -0.0056
        3h     94   0.8877    85.0%    62.6%    72.1%  -0.0059
        4h    125   0.8863    85.8%    61.9%    71.9%  -0.0073
        5h    157   0.8852    86.1%    61.2%    71.6%  -0.0084

Fallback AUC: 0.7360 (reference)


## 13. Save Production Models

In [39]:
# --- Save production model ---
VERSION = 'dep_v9_0'

if 'CatBoost' in best_model_name:
    production_model_obj = cat_tuned
    production_params = cat_best_params
    best_trial_params = study_cat.best_params
else:
    production_model_obj = lgb_tuned
    production_params = lgb_best_params
    best_trial_params = study_lgb.best_params

best_auc = model_aucs[best_model_name]

# Build temporal decay config
temporal_decay_config = {
    'use_temporal': best_trial_params.get('use_temporal', False),
    'half_life': best_trial_params.get('half_life', None),
}

# Target encoding maps (from train set)
train_delay_rate = float(y_train.mean())
gate_target = train.groupby('Gate')['Is_Delayed'].mean()
airline_target = train.groupby('Marketing Airline Desc')['Is_Delayed'].mean()
runway_target = train.groupby('Departure_Runway_Clean')['Is_Delayed'].mean() if 'Departure_Runway_Clean' in train.columns else pd.Series(dtype=float)

# Balanced mode threshold and metrics
prod_t, prod_m = find_optimal_threshold(y_test.values, best_proba, target_recall=0.65, min_precision=0.30)

# Save main model
model_path = MODELS_DIR / f'production_model_{VERSION}.joblib'
joblib.dump({
    'model': production_model_obj,
    'model_type': best_model_name,
    'best_params': production_params,
    'auc': best_auc,
    'optimal_threshold': prod_t,
    'metrics_at_threshold': prod_m,
    'feature_columns': feature_columns,
    'train_medians': train_medians.to_dict(),
    'gate_target_encoding': gate_target.to_dict(),
    'airline_target_encoding': airline_target.to_dict(),
    'runway_target_encoding': runway_target.to_dict(),
    'train_delay_rate': train_delay_rate,
    'temporal_decay': temporal_decay_config,
    'operating_modes': {
        name: {'threshold': r['threshold'], 'precision': r['precision'], 'recall': r['recall'], 'f1': r['f1']}
        for name, r in mode_results.items()
    },
    'version': VERSION,
}, model_path)
print(f'Saved: {model_path}')

# Save calibrator
cal_path = MODELS_DIR / f'calibrator_{VERSION}.joblib'
joblib.dump(production_calibrator, cal_path)
print(f'Saved: {cal_path}')

# Save fallback model
fb_t, fb_m = find_optimal_threshold(y_test.values, fb_final_proba, target_recall=0.65, min_precision=0.20)
fb_path = MODELS_DIR / f'fallback_model_{VERSION}.joblib'
joblib.dump({
    'model': fb_model,
    'model_type': f'{fb_model_name} Fallback',
    'auc': fb_final_auc,
    'feature_columns': fallback_features,
    'optimal_threshold': fb_t,
    'metrics_at_threshold': fb_m,
    'version': VERSION,
}, fb_path)
print(f'Saved: {fb_path}')

Saved: ../../../models/production_model_dep_v9_0.joblib
Saved: ../../../models/calibrator_dep_v9_0.joblib
Saved: ../../../models/fallback_model_dep_v9_0.joblib


### 13.1 Save Config

In [40]:
# --- Save configuration ---
config = {
    'model_type': best_model_name,
    'version': VERSION,
    'direction': 'departure',
    'auc': round(float(best_auc), 4),
    'feature_columns': feature_columns,
    'n_features': len(feature_columns),
    'classification': {
        'threshold': float(prod_t) if prod_t else 0.50,
        'precision': round(float(prod_m['precision']), 4) if prod_m else None,
        'recall': round(float(prod_m['recall']), 4) if prod_m else None,
        'f1': round(float(prod_m['f1']), 4) if prod_m else None,
        'fallback_auc': round(float(fb_final_auc), 4),
        'ece_raw': round(float(ece_raw), 4),
        'ece_calibrated': round(float(ece_cal), 4),
    },
    'risk_tiers_4': {
        'thresholds': list(best_4tier) if best_4tier else None,
        'separation': round(best_4tier_sep, 1),
    },
    'operating_modes': {
        name: {'threshold': r['threshold']}
        for name, r in mode_results.items()
    },
    'temporal_decay': temporal_decay_config,
    'train_info': {
        'n_train': int(len(train)),
        'n_test': int(len(test)),
        'train_delay_rate': round(float(y_train.mean()), 4),
        'test_delay_rate': round(float(y_test.mean()), 4),
    },
}
with open(OUTPUT_DIR / f'departure_config_{VERSION}.json', 'w') as f:
    json_mod.dump(config, f, indent=1, ensure_ascii=False)
print(f'Saved: departure_config_{VERSION}.json')

Saved: departure_config_dep_v9_0.json


### 13.2 Save Optuna Studies

In [41]:
# --- Save Optuna studies ---
study_path = MODELS_DIR / f'optuna_studies_{VERSION}.pkl'
with open(study_path, 'wb') as f:
    pickle.dump({
        'study_lgb': study_lgb,
        'study_xgb': study_xgb,
        'study_cat': study_cat,
    }, f)
print(f'Saved: {study_path}')

Saved: ../../../models/optuna_studies_dep_v9_0.pkl


In [42]:
# === Save classification context for downstream notebooks (04 regression) ===
dep_model_context = {
    'feature_columns': feature_columns,
    'X_train': X_train, 'X_test': X_test,
    'y_train': y_train, 'y_test': y_test,
    'train': train, 'test': test,
    'train_medians': train_medians,
    # Classification models & artifacts
    'production_model': production_model_obj,
    'lgb_model': lgb_tuned,
    'fallback_model': fb_model,
    'fallback_features': fallback_features,
    'calibrator': production_calibrator,
    'raw_proba': best_proba,
    'cal_proba_full': best_proba_calibrated,
    'lgb_proba': lgb_tuned_proba,
    # Metrics
    'auc_prod': best_auc,
    'lgb_auc': roc_auc_score(y_test, lgb_tuned_proba),
    'fb_auc': fb_final_auc,
    'optimal_threshold': optimal_threshold,
    'best_balanced': balanced_result,
    'best_hp': mode_results.get('High Precision', {}),
    'best_hr': mode_results.get('High Recall', {}),
    'ece_before': ece_raw,
    'ece_after': ece_cal,
    'best_3tier': best_3tier,
    'best_4tier': best_4tier,
}
pickle.dump(dep_model_context, open(DATA_PROCESSED / 'dep_deployment_context.pkl', 'wb'))
print(f'Saved dep_deployment_context.pkl (for NB04/06 downstream)')

Saved dep_deployment_context.pkl (for NB04/06 downstream)


## 14. Summary

In [44]:
# --- Final summary ---
print('\n' + '=' * 80)
print(f'DEPARTURE DEPLOYMENT SUMMARY (V9.0: {len(feature_columns)} features)')
print('=' * 80)

print(f'\n--- Classification ---')
print(f'  Best model:           {best_model_name}')
print(f'  AUC:                  {best_auc:.4f}')
if prod_t:
    print(f'  Production threshold: {prod_t:.2f}')
    print(f'  Precision:            {prod_m["precision"]:.1%}')
    print(f'  Recall:               {prod_m["recall"]:.1%}')
    print(f'  F1:                   {prod_m["f1"]:.1%}')
print(f'  Fallback AUC:         {fb_final_auc:.4f} ({fb_model_name})')
print(f'  Calibration ECE:      {ece_raw:.4f} → {ece_cal:.4f}')

print(f'\n--- Operating Modes ---')
for mode_name, r in mode_results.items():
    m = r
    print(f'  {mode_name:20s}: t={r["threshold"]:.2f}, P={m["precision"]:.1%}, R={m["recall"]:.1%}')

if best_4tier:
    crit_t, high_t, med_t = best_4tier
    r = best_4tier_results
    print(f'\n--- 4-Tier Risk (sep={best_4tier_sep:.1f}x) ---')
    print(f'  CRITICAL ≥{crit_t:.2f}: {r["CRITICAL"]["delay_rate"]:.1%}')
    print(f'  HIGH     ≥{high_t:.2f}: {r["HIGH"]["delay_rate"]:.1%}')
    print(f'  MEDIUM   ≥{med_t:.2f}: {r["MEDIUM"]["delay_rate"]:.1%}')
    print(f'  LOW      <{med_t:.2f}: {r["LOW"]["delay_rate"]:.1%}')

print(f'\n--- Lookahead ---')
for r in lookahead_results:
    print(f'  {r["lookahead_h"]}h: AUC={r["auc"]:.4f} (Δ={r["auc"]-base_auc:+.4f})')

print(f'\n--- Baselines vs Tuned ---')
for name in ['LGB baseline', 'XGB baseline', 'CatBoost baseline', 'LGB Optuna', 'XGB Optuna', 'CatBoost Optuna']:
    print(f'  {name:<25s} AUC={model_aucs[name]:.4f}')

print(f'\n--- Saved Files ---')
print(f'  production_model_{VERSION}.joblib')
print(f'  calibrator_{VERSION}.joblib')
print(f'  fallback_model_{VERSION}.joblib')
print(f'  optuna_studies_{VERSION}.pkl')
print(f'  departure_config_{VERSION}.json')
print(f'  dep_baseline_context.pkl')
print(f'\n--- V8.0 Reference ---')
print(f'  V8.0 AUC: 0.8917 | V9.0: {best_auc:.4f} (Δ={best_auc-0.8917:+.4f})')
print(f'\nDone!')


DEPARTURE DEPLOYMENT SUMMARY (V9.0: 23 features)

--- Classification ---
  Best model:           CatBoost Optuna
  AUC:                  0.8936
  Production threshold: 0.59
  Precision:            77.4%
  Recall:               68.0%
  F1:                   72.4%
  Fallback AUC:         0.7360 (CatBoost)
  Calibration ECE:      0.1672 → 0.0677

--- Operating Modes ---
  High Precision      : t=0.79, P=88.9%, R=60.3%
  High Recall         : t=0.17, P=30.8%, R=90.5%
  Balanced (Default)  : t=0.68, P=82.5%, R=65.2%

--- 4-Tier Risk (sep=30.0x) ---
  CRITICAL ≥0.90: 98.9%
  HIGH     ≥0.20: 45.5%
  MEDIUM   ≥0.05: 8.8%
  LOW      <0.05: 3.3%

--- Lookahead ---
  0h: AUC=0.8936 (Δ=+0.0000)
  1h: AUC=0.8909 (Δ=-0.0027)
  2h: AUC=0.8880 (Δ=-0.0056)
  3h: AUC=0.8877 (Δ=-0.0059)
  4h: AUC=0.8863 (Δ=-0.0073)
  5h: AUC=0.8852 (Δ=-0.0084)

--- Baselines vs Tuned ---
  LGB baseline              AUC=0.8808
  XGB baseline              AUC=0.8801
  CatBoost baseline         AUC=0.8868
  LGB Optuna     